# Ask-SLM Runner
Minimal notebook flow: aggregate view inputs, prepare mappings, build analysis-wide dataset, and launch managers.

In [8]:
from pathlib import Path
import os
import sys

import pandas as pd

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from managers.runner_pipeline_manager import (
    build_or_load_tax_accessor,
    enrich_envo_columns,
    initialize_runner_context,
    prepare_workspace_inputs,
    refresh_mapping_templates,
    run_enrichment_stage,
    run_merge_stage,
    run_taxonomy_stage,
)


def require_vars(names):
    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError(f"Missing required variables: {', '.join(missing)}")

In [9]:
workspace = "ABX"

context, runner_logger = initialize_runner_context(workspace=workspace)

# Convenience aliases for downstream cells.
results_dir = context.results_dir
config_dir = context.config_dir


def log_run(message: str, echo: bool = False):
    runner_logger.log(message, echo=echo)

[2026-04-19 14:20:27] Log file path: Workspaces/ABX/Logs/run_20260419_142027_597018.log


In [10]:
metadata_table, taxonomy_table, metadata_source, taxonomy_source = prepare_workspace_inputs(
    context=context,
    logger=runner_logger,
)

display(metadata_table.head())

,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,ena_first_public_run,ena_last_update_run,sample_name_sam,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject
1284,SRR29697434,AMPLICON,STANFORD,public,SRX25200342,BAD_39,Illumina NovaSeq X,PAIRED,PCR,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=1824020400}, {k=bytes, v=53705176...","{""bases"": 1824020400, ""bytes"": 537051765, ""run...",1,PRJNA974858
1285,SRR24705147,WGS,STANFORD,public,SRX20483864,EAC_50,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=11618174256}, {k=bytes, v=3730732...","{""bases"": 11618174256, ""bytes"": 3730732746, ""r...",1,PRJNA974858
1286,SRR24705364,WGS,STANFORD,public,SRX20484319,EAQ_57,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=4701894376}, {k=bytes, v=15178353...","{""bases"": 4701894376, ""bytes"": 1517835301, ""ru...",1,PRJNA974858
1287,SRR24705548,WGS,STANFORD,public,SRX20484135,EAP_63,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=6101445704}, {k=bytes, v=19618863...","{""bases"": 6101445704, ""bytes"": 1961886381, ""ru...",1,PRJNA974858
1288,SRR24705617,WGS,STANFORD,public,SRX20484066,BAM_10,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=6404763040}, {k=bytes, v=20559767...","{""bases"": 6404763040, ""bytes"": 2055976750, ""ru...",1,PRJNA974858


In [11]:
sra_processor, project_ids, project_status_df = refresh_mapping_templates(
    metadata_df=metadata_table,
    context=context,
    logger=runner_logger,
)

display(project_status_df)

,project_id,config_exists,config_path
0,PRJNA974858,True,Workspaces/ABX/mapping_configs/PRJNA974858.json


In [12]:
from Engines.MappingConfigEngine.mapping_config_engine import MappingConfigManager

mapping_manager = MappingConfigManager(
    config_dir=context.config_dir,
    logger=runner_logger.log,
)
mapping_manager.run()

✅ Loaded 27 fields from Metalog schema
📋 Loaded 1 project(s)



In [13]:
enriched_metadata, mapped_cols, missing_mapping_projects, enrichment_current_stamp = run_enrichment_stage(
    metadata_df=metadata_table,
    processor=sra_processor,
    project_ids=project_ids,
    context=context,
    logger=runner_logger,
    strict_missing_configs=False,
)

enriched_metadata, envo_report_df = enrich_envo_columns(
    metadata_df=enriched_metadata,
    context=context,
    logger=runner_logger,
)

summary_df = pd.DataFrame(
    {
        "metric": ["samples", "projects", "mapped_columns", "projects_missing_mapping_config", "envo_terms_resolved"],
        "value": [
            len(enriched_metadata),
            enriched_metadata["bioproject"].nunique(),
            len(mapped_cols),
            len(missing_mapping_projects),
            len(envo_report_df),
        ],
    }
)

display(summary_df)

if missing_mapping_projects:
    display(
        pd.DataFrame(
            {
                "missing_project_id": missing_mapping_projects,
                "expected_config_path": [str(context.config_dir / f"{p}.json") for p in missing_mapping_projects],
                "template_path": [str(context.config_dir / f"{p}.json.template") for p in missing_mapping_projects],
            }
        )
    )

if not envo_report_df.empty:
    display(envo_report_df.head(20))

display(enriched_metadata.head())

[2026-04-19 14:20:35] Enrichment cache miss: rebuilt and saved to Workspaces/ABX/results/_cache


,metric,value
0,samples,960
1,projects,1
2,mapped_columns,0
3,projects_missing_mapping_config,0
4,envo_terms_resolved,0


,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,ena_first_public_run,ena_last_update_run,sample_name_sam,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject
0,SRR24705147,WGS,STANFORD,public,SRX20483864,EAC_50,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=11618174256}, {k=bytes, v=3730732...","{""bases"": 11618174256, ""bytes"": 3730732746, ""r...",1,PRJNA974858
1,SRR24705364,WGS,STANFORD,public,SRX20484319,EAQ_57,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=4701894376}, {k=bytes, v=15178353...","{""bases"": 4701894376, ""bytes"": 1517835301, ""ru...",1,PRJNA974858
2,SRR24705548,WGS,STANFORD,public,SRX20484135,EAP_63,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=6101445704}, {k=bytes, v=19618863...","{""bases"": 6101445704, ""bytes"": 1961886381, ""ru...",1,PRJNA974858
3,SRR24705617,WGS,STANFORD,public,SRX20484066,BAM_10,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=6404763040}, {k=bytes, v=20559767...","{""bases"": 6404763040, ""bytes"": 2055976750, ""ru...",1,PRJNA974858
4,SRR24705539,WGS,STANFORD,public,SRX20484144,EAP_75,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=17239632600}, {k=bytes, v=5449924...","{""bases"": 17239632600, ""bytes"": 5449924461, ""r...",1,PRJNA974858


In [14]:
TAXON_KEY_SEP = "||"

abundance_matrix, taxon_lookup_df, current_taxonomy_stamp = run_taxonomy_stage(
    taxonomy_df=taxonomy_table,
    context=context,
    logger=runner_logger,
    taxon_key_sep=TAXON_KEY_SEP,
)

abundance_metadata_matrix, metadata_columns, current_merge_stamp = run_merge_stage(
    enriched_metadata_df=enriched_metadata,
    abundance_matrix=abundance_matrix,
    enrichment_stamp=enrichment_current_stamp,
    taxonomy_stamp=current_taxonomy_stamp,
    context=context,
    logger=runner_logger,
)

artifacts_df = pd.DataFrame(
    {
        "artifact": [
            "abundance_matrix",
            "abundance_metadata_matrix",
            "taxon_lookup",
            "metadata_columns",
            "taxonomy_stage_stamp",
            "merge_stage_stamp",
            "combined_stamp",
            "envo_dictionary",
        ],
        "path": [
            str(context.results_dir / "Ask_SLM_Taxon_Abundance_Matrix.csv"),
            str(context.results_dir / "Ask_SLM_Analysis_Wide_Dataset.csv"),
            str(context.results_dir / "Ask_SLM_Taxon_Lookup.csv"),
            str(context.results_dir / "Ask_SLM_Metadata_Columns.json"),
            str(context.cache_dir / "taxonomy_stage_stamp.json"),
            str(context.cache_dir / "merge_stage_stamp.json"),
            str(context.results_dir / "Ask_SLM_Cache_Stamp.json"),
            str(context.results_dir / "ENVO_Term_Dictionary.csv"),
        ],
    }
)

display(artifacts_df)
display(abundance_metadata_matrix[metadata_columns].head())

[2026-04-19 14:20:46] Taxonomy cache miss: rebuilt taxonomy features in Workspaces/ABX/results
[2026-04-19 14:21:21] Merge cache miss: rebuilt merged dataset in Workspaces/ABX/results


,artifact,path
0,abundance_matrix,Workspaces/ABX/results/Ask_SLM_Taxon_Abundance...
1,abundance_metadata_matrix,Workspaces/ABX/results/Ask_SLM_Analysis_Wide_D...
2,taxon_lookup,Workspaces/ABX/results/Ask_SLM_Taxon_Lookup.csv
3,metadata_columns,Workspaces/ABX/results/Ask_SLM_Metadata_Column...
4,taxonomy_stage_stamp,Workspaces/ABX/results/_cache/taxonomy_stage_s...
5,merge_stage_stamp,Workspaces/ABX/results/_cache/merge_stage_stam...
6,combined_stamp,Workspaces/ABX/results/Ask_SLM_Cache_Stamp.json
7,envo_dictionary,Workspaces/ABX/results/ENVO_Term_Dictionary.csv


,acc,assay_type,center_name,consent,experiment,sample_name,instrument,librarylayout,libraryselection,librarysource,...,ena_first_public_run,ena_last_update_run,sample_name_sam,datastore_filetype,datastore_provider,datastore_region,attributes,jattr,run_file_version,bioproject
0,SRR24705147,WGS,STANFORD,public,SRX20483864,EAC_50,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=11618174256}, {k=bytes, v=3730732...","{""bases"": 11618174256, ""bytes"": 3730732746, ""r...",1,PRJNA974858
1,SRR24705364,WGS,STANFORD,public,SRX20484319,EAQ_57,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=4701894376}, {k=bytes, v=15178353...","{""bases"": 4701894376, ""bytes"": 1517835301, ""ru...",1,PRJNA974858
2,SRR24705548,WGS,STANFORD,public,SRX20484135,EAP_63,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=6101445704}, {k=bytes, v=19618863...","{""bases"": 6101445704, ""bytes"": 1961886381, ""ru...",1,PRJNA974858
3,SRR24705617,WGS,STANFORD,public,SRX20484066,BAM_10,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=6404763040}, {k=bytes, v=20559767...","{""bases"": 6404763040, ""bytes"": 2055976750, ""ru...",1,PRJNA974858
4,SRR24705539,WGS,STANFORD,public,SRX20484144,EAP_75,Illumina NovaSeq 6000,PAIRED,RANDOM,GENOMIC,...,NaN,NaN,NaN,"[fastq, run.zq, sra]","[gs, ncbi, s3]","[gs.us-east1, ncbi.public, s3.us-east-1]","[{k=bases, v=17239632600}, {k=bytes, v=5449924...","{""bases"": 17239632600, ""bytes"": 5449924461, ""r...",1,PRJNA974858


In [15]:
from Engines.AlphaDiversityEngine.alpha_diversity_engine import AlphaDiversityManager

require_vars(["abundance_metadata_matrix", "taxon_lookup_df", "metadata_columns", "current_taxonomy_stamp"])

shared_tax_accessor = build_or_load_tax_accessor(
    taxon_lookup_df=taxon_lookup_df,
    taxonomy_stamp=current_taxonomy_stamp,
    context=context,
    logger=runner_logger,
)


Building lineage database from 23283 rows...
✅ Built lineage database with 23283 unique taxa
✅ Saved lineage database to Workspaces/ABX/results/_cache/taxonomy_lineage_db.json
[2026-04-19 14:21:38] Taxonomy accessor cache miss: rebuilt lineage DB at Workspaces/ABX/results/_cache/taxonomy_lineage_db.json


In [16]:

alpha_manager = AlphaDiversityManager(
    analysis_wide_df=abundance_metadata_matrix,
    taxon_lookup_df=taxon_lookup_df,
    metadata_cols=metadata_columns,
    tax_accessor=shared_tax_accessor,
    logger=runner_logger.log,
)
alpha_manager.run()

KeyboardInterrupt: 

In [ ]:
from Engines.TaxonomicAbundanceEngine import TaxonomicAbundanceManager

require_vars(["abundance_metadata_matrix", "taxon_lookup_df", "metadata_columns"])

abundance_manager = TaxonomicAbundanceManager(
    abundance_metadata_matrix=abundance_metadata_matrix,
    taxon_lookup_df=taxon_lookup_df,
    metadata_cols=metadata_columns,
    logger=runner_logger.log,
)
abundance_manager.run()

In [ ]:
# from Engines.LongitudinalAnalysisEngine import LongitudinalAnalysisManager

# require_vars(["enriched_df", "sra_metadata_df"])

# longitudinal_manager = LongitudinalAnalysisManager(enriched_metadata, set(metadata_table.columns))
# longitudinal_manager.run()